# 07 — Validate and Merge All Datasets

**Owner:** Mostafa Elsherif
**Input:** the 8 processed files from notebooks 00-06
**Output:** `data/processed/master_monthly.csv` — 60,480 rows, keys = `cell_id, year, month`

## Why we need this

Our 8 datasets come from NASA and FAO. They have different resolutions (10 m to 36 km), different time steps (daily, monthly, yearly, fixed) and different shapes. No model can read 8 separate files.

We join them into **one table** where every row is "one 1 km cell in one month":

**560 cells x 108 months = 60,480 rows.**

## Step 1 — Validate before merging

Every file is checked against the contract in `data_workflow.md` **before** we touch it: row count, key columns, duplicate keys, year range, fill values, missing values.

Finding a problem here takes seconds. Finding it after the merge takes hours.

## Step 2 — Merge safely

Sentinel-2 is the **base table**, because it already has the exact shape we want. Everything else is joined onto it with a `left` join, so the row count can never change.

Each join runs 4 checks:

1. the right table has **no duplicate keys** - a duplicate would silently multiply our rows
2. **no column name clash** - a clash would silently rename columns to `_x` and `_y`
3. the **row count did not change** after the join
4. the **match rate** is reported, so we see if a file does not cover all our cells or months

Then 6 final checks on the result.

## What each file brings

| File | Shape | What it adds |
|---|---|---|
| `sentinel2_monthly` | cell x month | NDVI, bare soil %, weak cover %, farmland % |
| `grid_1km` | cell | position (lat, lon, UTM) |
| `srtm_static` | cell | elevation |
| `gaez_static` | cell | suitability, class and yield for 8 crops |
| `power_monthly` | month | temperature, humidity, wind, solar |
| `gpm_monthly` | month | rainfall, rain days, heaviest day |
| `smap_monthly` | month | rainfed soil moisture, wilting fraction |
| `landsat_yearly` | cell x year | 40-year NDVI history |

In [16]:
import os
import numpy as np, pandas as pd

from pathlib import Path
REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent          # works for everyone, not just one laptop
P    = os.path.join(REPO, "data", "processed")
N_CELLS, N_MONTHS = 560, 108

CONTRACT = {
    "grid_1km.csv":          (["cell_id"], 560),
    "power_monthly.csv":     (["year", "month"], 108),
    "gpm_monthly.csv":       (["year", "month"], 108),
    "smap_monthly.csv":      (["year", "month"], 108),
    "sentinel2_monthly.csv": (["cell_id", "year", "month"], 60480),
    "srtm_static.csv":       (["cell_id"], 560),
    "gaez_static.csv":       (["cell_id"], 560),
    "landsat_yearly.csv":    (["cell_id", "year"], 22400),
}

for fname, (keys, n_rows) in CONTRACT.items():
    path = os.path.join(P, fname)
    if not os.path.exists(path):
        print(f"MISSING   {fname}")
        continue
    df = pd.read_csv(path)
    problems = []
    if len(df) != n_rows:
        problems.append(f"rows {len(df)} (want {n_rows})")
    miss_keys = [k for k in keys if k not in df.columns]
    if miss_keys:
        problems.append(f"no key {miss_keys}")
    elif df.duplicated(keys).any():
        problems.append("duplicate keys")
    if "year" in df.columns and fname != "landsat_yearly.csv":
        bad = df[~df.year.between(2016, 2024)]
        if len(bad):
            problems.append(f"years outside 2016-2024: {sorted(bad.year.unique())}")
    num = df.select_dtypes("number")
    fills = int((num <= -999).sum().sum())
    if fills:
        problems.append(f"{fills} fill values")
    miss_pct = df.isna().mean().max() * 100

    flag = "OK  " if not problems else "BAD "
    print(f"{flag}{fname:24s} rows={len(df):6d}  maxmissing={miss_pct:5.1f}%  {'; '.join(problems)}")

OK  grid_1km.csv             rows=   560  maxmissing=  0.0%  
OK  power_monthly.csv        rows=   108  maxmissing=  0.0%  
OK  gpm_monthly.csv          rows=   108  maxmissing=  0.0%  
OK  smap_monthly.csv         rows=   108  maxmissing=  0.0%  
OK  sentinel2_monthly.csv    rows= 60480  maxmissing=  0.8%  
OK  srtm_static.csv          rows=   560  maxmissing=  0.0%  
OK  gaez_static.csv          rows=   560  maxmissing=  7.1%  
OK  landsat_yearly.csv       rows= 22400  maxmissing=  1.4%  


In [17]:
def load(name):
    path = os.path.join(P, name)
    if not os.path.exists(path):
        print(f"  (missing) {name}")
        return None
    df = pd.read_csv(path)
    print(f"  loaded   {name:24s} {df.shape}")
    return df

print("LOADING")
s2    = load("sentinel2_monthly.csv")
grid  = load("grid_1km.csv")
srtm  = load("srtm_static.csv")
power = load("power_monthly.csv")
gpm   = load("gpm_monthly.csv")
smap  = load("smap_monthly.csv")
ls    = load("landsat_yearly.csv")
gaez  = load("gaez_static.csv")


def safe_merge(left, right, keys, label):
    if right is None:
        print(f"SKIP   {label:22s} (file not available)")
        return left
    assert not right.duplicated(keys).any(), f"{label}: duplicate keys"
    before  = len(left)
    overlap = (set(left.columns) & set(right.columns)) - set(keys)
    assert not overlap, f"{label}: column name clash {overlap}"
    out = left.merge(right, on=keys, how="left")
    assert len(out) == before, f"{label}: rows changed {before} -> {len(out)}"
    new_cols = [c for c in right.columns if c not in keys]
    matched  = out[new_cols[0]].notna().mean() * 100
    print(f"OK     {label:22s} rows={len(out):6d}  matched={matched:5.1f}%")
    return out


master = s2.copy()
assert len(master) == N_CELLS * N_MONTHS, f"base rows {len(master)}"
print(f"\nBASE   sentinel2_monthly    rows={len(master)}")

# static, per cell
master = safe_merge(master, grid[["cell_id", "lat", "lon", "x_m", "y_m"]], ["cell_id"], "grid")
master = safe_merge(master, srtm, ["cell_id"], "srtm")
master = safe_merge(master, gaez, ["cell_id"], "gaez")

# whole area, monthly
master = safe_merge(master, power, ["year", "month"], "power")
master = safe_merge(master, gpm,   ["year", "month"], "gpm")
master = safe_merge(master, smap,  ["year", "month"], "smap")

# per cell, yearly (renamed to avoid clashing with Sentinel-2)
ls2 = ls.rename(columns={
    "ndvi_max_mean":    "ls_ndvi_max",
    "ndvi_winter_mean": "ls_ndvi_winter",
    "n_images":         "ls_n_images",
    "n_winter":         "ls_n_winter",
    "farm_pct":         "ls_farm_pct",
})
master = safe_merge(master, ls2, ["cell_id", "year"], "landsat")

print("\nFINAL CHECKS")
checks = {
    "rows == 60480":            len(master) == N_CELLS * N_MONTHS,
    "unique cells == 560":      master.cell_id.nunique() == N_CELLS,
    "unique year-month == 108": len(master[["year", "month"]].drop_duplicates()) == N_MONTHS,
    "no duplicate keys":        not master.duplicated(["cell_id", "year", "month"]).any(),
    "years inside 2016-2024":   master.year.between(2016, 2024).all(),
    "no all-empty column":      not (master.isna().all()).any(),
}
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")

master = master.sort_values(["cell_id", "year", "month"]).reset_index(drop=True)
OUT = os.path.join(P, "master_monthly.csv")
master.to_csv(OUT, index=False)

print(f"\nsaved: {OUT}")
print(f"shape: {master.shape}")
print("\nmissing % per column (top 10):")
print((master.isna().mean() * 100).round(2).sort_values(ascending=False).head(10))

LOADING
  loaded   sentinel2_monthly.csv    (60480, 8)
  loaded   grid_1km.csv             (560, 11)
  loaded   srtm_static.csv          (560, 4)
  loaded   power_monthly.csv        (108, 9)
  loaded   gpm_monthly.csv          (108, 5)
  loaded   smap_monthly.csv         (108, 8)
  loaded   landsat_yearly.csv       (22400, 7)
  loaded   gaez_static.csv          (560, 25)

BASE   sentinel2_monthly    rows=60480
OK     grid                   rows= 60480  matched=100.0%
OK     srtm                   rows= 60480  matched=100.0%
OK     gaez                   rows= 60480  matched= 92.9%
OK     power                  rows= 60480  matched=100.0%
OK     gpm                    rows= 60480  matched=100.0%
OK     smap                   rows= 60480  matched=100.0%
OK     landsat                rows= 60480  matched=100.0%

FINAL CHECKS
  PASS  rows == 60480
  PASS  unique cells == 560
  PASS  unique year-month == 108
  PASS  no duplicate keys
  PASS  years inside 2016-2024
  PASS  no all-empty colum

## Conclusion

All 8 files passed validation with **zero problems**. The merge produced **60,480 rows x 60 columns**, and every check passed:

| Check | Result |
|---|---|
| rows == 60,480 | PASS |
| unique cells == 560 | PASS |
| unique year-month == 108 | PASS |
| no duplicate keys | PASS |
| years inside 2016-2024 | PASS |
| no all-empty column | PASS |

### Match rates

Seven of the eight joins matched **100%** of rows: no file is missing a cell, a month or a year.

GAEZ matched **92.9%**. That is not a merge failure. It is exactly the 40 cells out of 560 (7.1%) where FAO has no suitability value for most crops, explained in notebook 06. The merge is working correctly and is reporting a real gap in the source data instead of hiding it.

### Missing values

| Columns | Missing | Why |
|---|---|---|
| `si_*` for 6 crops | 7.14% | The 40 cells FAO has no value for. Left empty, not filled. |
| `ndvi_mean`, `bare_pct`, `low_cover_pct` | 0.77% | Cell-months where Sentinel-2 had no cloud-free pixel at all. |

Both gaps are honest and documented. We would rather carry an empty value than invent a number. The `valid_pct` column records how much clear satellite data each cell-month had, so any model we build can weight or drop low-quality rows instead of guessing.

### What this unlocks

`master_monthly.csv` is now the single input for everything that follows: cover-gap signals, the crop table, rotation scoring, and the web tool. No later step needs to open a raw dataset again.